
# Inverting a Cooled Source: Diggins26 with Cooling Branch Selection

**Objective:** show that
:func:`~trilobite.radiation.synchrotron.spectral_inversion.invert_ssa_with_cooling_Diggins26`
recovers $B$ and $R$ whether the electrons at the SSA break are uncooled, cooled by synchrotron
losses, or cooled by inverse Compton (IC) losses, and that ignoring cooling biases the result.

<div class="alert alert-info"><h4>Note</h4>

The analytic one-zone SED classes are being phased out in favor of the numerical engines, which
are more accurate. Here the SED is generated with the numerical engine, the break is extracted by a
fit, and only then is it inverted. The inversion is therefore tested against an independent calculation,
not against its own forward model.

</div>

Cooling steepens the spectrum and shifts the SSA break, so the uncooled inversion
(:func:`~trilobite.radiation.synchrotron.spectral_inversion.invert_ssa_Diggins26`) is wrong when
$\nu_c < \nu_a$. The cooling-aware function needs two extra inputs, the source age $t$ and the
seed-photon luminosity $L$ for IC losses, and returns the branch (cooled or uncooled) that is
self-consistent.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from astropy import units as u
from scipy.optimize import curve_fit

from trilobite.radiation.synchrotron import invert_ssa_Diggins26, invert_ssa_with_cooling_Diggins26
from trilobite.radiation.synchrotron.cooling import InverseComptonCoolingEngine, SynchrotronRadiativeCoolingEngine
from trilobite.radiation.synchrotron.electron_distributions import PowerLaw
from trilobite.radiation.synchrotron.SEDs.numerical import NonRelativisticSphericalSynchrotronEngine
from trilobite.utils.plot_utils import set_plot_style

set_plot_style()

## Three Sources

All three are supernova shells at 20 Mpc with $p = 3$. They differ in field, size, age and
photospheric luminosity:

- **Uncooled:** a weakly magnetized shell, where the cooling break lies far above the SSA peak.
- **Synchrotron-cooled:** a strongly magnetized shell in dense circumstellar material, observed at 100 days.
- **IC-cooled:** a young shell close to a bright photosphere, $L = 10^{43}$ erg/s.

The numerical SED uses a cooled electron distribution: $N \propto \gamma^{-p}$ below
$\gamma_c$ and $\propto \gamma^{-(p+1)}$ above it, with $\gamma_c$ from the
Trilobite cooling engines.



In [ ]:
D, p = 20.0 * u.Mpc, 3.0
micro = dict(epsilon_E=0.1, epsilon_B=0.1, gamma_min=1.0, gamma_max=1e6)
geometry = dict(f_A=0.5, f_V=0.5)

sources = {
    "Uncooled": dict(B=1.0 * u.G, R=1.0e16 * u.cm, t=30 * u.day, L=0.0 * u.erg / u.s),
    "Synchrotron-cooled": dict(B=5.0 * u.G, R=5.0e15 * u.cm, t=100 * u.day, L=0.0 * u.erg / u.s),
    "IC-cooled": dict(B=3.0 * u.G, R=3.0e15 * u.cm, t=30 * u.day, L=1.0e43 * u.erg / u.s),
}

engine = NonRelativisticSphericalSynchrotronEngine()
engine.load_avg_first_kernel()
sync_engine, ic_engine = SynchrotronRadiativeCoolingEngine(), InverseComptonCoolingEngine()

nu = np.geomspace(0.1, 300.0, 60) * u.GHz
gamma = np.geomspace(micro["gamma_min"], micro["gamma_max"], 600)


def numerical_sed(B, R, t, L):
    """Numerical flux density of a source with synchrotron and IC cooling."""
    inv_gamma_c = 1.0 / sync_engine.compute_cooling_gamma(B=B, t=t)
    if L > 0:
        inv_gamma_c += 1.0 / ic_engine.compute_cooling_gamma(L_bol=L, R=R, t=t)
    gamma_c = 1.0 / inv_gamma_c

    N0 = PowerLaw.normalize_from_magnetic_field(B, p=p, **micro).to_value(u.cm**-3)
    N = N0 * np.where(gamma < gamma_c, gamma**-p, gamma_c * gamma ** (-p - 1))
    return engine.compute_flux_density(nu, R=R, B=B, N=N, gamma=gamma, luminosity_distance=D, **geometry).to(u.mJy)

## Extracting the Break

As before, we fit a smoothly broken power law with a $\nu^{5/2}$ thick slope and a free thin slope.
The fitted intersection of the asymptotes is $(\nu_{\rm brk}, F_{\rm brk})$.



In [ ]:
def sbpl(log_nu, log_F, log_nu_brk, beta, s):
    x = (log_nu - log_nu_brk) * np.log(10)
    return log_F + s * np.logaddexp(2.5 * x / s, beta * x / s) / np.log(10)


def extract_break(flux):
    """Fit the SED and return ``(nu_brk, F_brk, fit_parameters)``."""
    log_nu, log_F = np.log10(nu.to_value(u.Hz)), np.log10(flux.to_value(u.Jy))
    guess = [log_F.max(), log_nu[np.argmax(log_F)], -1.0, -1.0]
    fit, _ = curve_fit(sbpl, log_nu, log_F, p0=guess, bounds=([-20, 7, -4, -10], [5, 12, 0, -0.05]))
    return 10 ** fit[1] * u.Hz, 10 ** fit[0] * u.Jy, fit

## Inverting With and Without Cooling

For each source we invert the extracted break twice: with the uncooled inversion, and with the cooling-aware
inversion given the true age and luminosity.



In [ ]:
results = {}
for name, src in sources.items():
    flux = numerical_sed(**src)
    nu_brk, F_brk, fit = extract_break(flux)
    naive = invert_ssa_Diggins26(nu_brk, F_brk, D, p=p, pitch_average=True, **micro, **geometry)
    aware = invert_ssa_with_cooling_Diggins26(
        nu_brk, F_brk, D, src["t"], src["L"], p=p, pitch_average=True, **micro, **geometry
    )
    results[name] = dict(flux=flux, nu_brk=nu_brk, F_brk=F_brk, fit=fit, naive=naive, aware=aware)

fig, axes = plt.subplots(1, 3, figsize=(13, 3.8), sharey=True, constrained_layout=True)
for ax, (name, res) in zip(axes, results.items()):
    ax.loglog(nu.to_value(u.GHz), res["flux"].to_value(u.mJy), "o", ms=3.5)
    ax.loglog(nu.to_value(u.GHz), 10 ** sbpl(np.log10(nu.to_value(u.Hz)), *res["fit"]) * 1e3, "k-", lw=1.2)
    ax.plot(res["nu_brk"].to_value(u.GHz), res["F_brk"].to_value(u.mJy), "r*", ms=12)
    ax.set(xlabel="Frequency [GHz]", title=f"{name}: branch = {res['aware']['branch']}")
axes[0].set_ylabel("Flux density [mJy]")
plt.show()

The function identifies the branch correctly in all three cases. The thin slope steepens by
$1/2$ when the electrons at the break are cooled ($F_\nu \propto \nu^{-p/2}$ instead of
$\nu^{-(p-1)/2}$).

## Recovery of the Source Properties



In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8), sharey=True, constrained_layout=True)
x = np.arange(len(sources))
for ax, key, truth, label in zip(axes, ["B", "R"], ["B", "R"], [r"$B$", r"$R$"]):
    naive = [(res["naive"][key] / sources[n][truth]).decompose().value for n, res in results.items()]
    aware = [(res["aware"][key] / sources[n][truth]).decompose().value for n, res in results.items()]
    ax.plot(x, naive, "s", ms=9, label="Uncooled inversion")
    ax.plot(x, aware, "o", ms=9, label="Cooling-aware inversion")
    ax.axhline(1.0, color="k", lw=1)
    ax.set(xticks=x, xticklabels=list(sources), ylabel=f"Recovered / true {label}", xlim=(-0.5, 2.5), ylim=(0.6, 1.1))
axes[0].legend(loc="lower left", fontsize=9)
plt.show()

**Outcome:** when the source is uncooled both inversions agree. When cooling matters, ignoring it
underestimates $B$ by about 30% and $R$ by about 20%, while the cooling-aware inversion recovers
both to within about 6%.

